Machine Learning Exercise 2: Classification Basics
In this exercise, you'll be working with a dataset containing information about passengers on the Titanic. You can get more information about this dataset here. Our goal will be to build a model to predict whether a passenger survived the voyage.

Let's start by building a model using Age as a predictor variable.

When performing a train/test split, you usually want to make sure that the ratios of target values look the same in the training set and in the test set. This is especially true with imbalanced targets, where there are more observations in one class than the other. This can be done by, instead of a completely random split, by performing a stratified split.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report, RocCurveDisplay

In [ ]:
titanic = pd.read_csv("data/titanic.csv")

In [ ]:
titanic.head()

In [ ]:
titanic.shape

Q1. Create a DataFrame, X, which contains one column, Age, and a Series, y, which contains the target variable. Perform a 70/30 train/test split, stratifying by the target variable. Make sure that you set a random_state during this split.

In [ ]:
# Feature and target
X = titanic[['Age']]
y = titanic['Survived']

# Train/test split (70/30 with stratification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=300,
    stratify=y
)

type(X)
type(y)

The target variable `y` is created as a pandas Series by selecting the `Survived` column using single brackets. This ensures that the target is a 1D structure, which is required for classification models in scikit-learn.

Q2. Now, try and fit a logistic regression model using the LogisticRegression class. When you do this, you're likely to encounter an error, that "Input X contains NaN." If you check, you'll find that there are some missing values in the Age column. In the next step, we'll address this issue.

In [ ]:
# model
model = LogisticRegression()

# fit model
model.fit(X_train, y_train)

In [ ]:
X_train.isna().sum()

The model failed because the Age feature has 129 missing values in the training set, and Logistic Regression cannot handle missing data directly

One option when a column contains missing values is to impute them, meaning to replace the missing values with an estimate. A simple way to impute is to fill missing values using the mean value. Important Note: When imputing with the mean, you should calculate the mean value only on the training data. In order to have an unbiased estimate of model performance, it is important that the model has access to no information about the test set.

Q3. Using a SimpleImputer, fit the training data and then use it to transform both the training and test data.



In [ ]:
# Create imputer with mean strategy
imputer = SimpleImputer(strategy='mean')

# Fit on training data only
X_train_imputed = imputer.fit_transform(X_train)

# Transform test data using the same imputer
X_test_imputed = imputer.transform(X_test)

In [ ]:
np.isnan(X_train_imputed).sum()
np.isnan(X_test_imputed).sum()

Q4. Now, fit your logistic regression model to the training data and generate predictions on the test data.

In [ ]:
# model
model = LogisticRegression()

# Fit model on imputed training data
model.fit(X_train_imputed, y_train)

# Generate predictions on test data
y_pred = model.predict(X_test_imputed)

In [ ]:
y_pred[:10]

The model predictions show that all passengers are classified as 0 (did not survive). This suggests that the model is not effectively distinguishing between survivors and non-survivors using the `Age` feature alone.

This may be due to the limited predictive power of age and the imbalance in the dataset, where more passengers did not survive than survived.

There are multiple ways we can assess the performance of a classification model. A common way is put those predictions into a confusion matrix, which compares the predicted and actual values.

Q5. Create a confusion matrix out of the predictions. What do you find?

In [ ]:
# Create confusion matrix
confuse_m = confusion_matrix(y_test, y_pred)

confuse_m

The confusion matrix shows that the model predicted all passengers as not surviving (0). It correctly classified 165 non-survivors (true negatives), but failed to identify any survivors, resulting in 103 false negatives and 0 true positives.

This indicates that the model is not effective at predicting survival when using Age as the only feature, as it does not capture meaningful patterns to distinguish survivors from non-survivors.

Q6. Plot the ROC curve. How well does your model do? Hint: You can use the RocCurveDisplay to create this plot.

In [ ]:
# Get predicted probabilities for class 1 (survived)
y_prob = model.predict_proba(X_test_imputed)[:, 1]

# Plot ROC curve
RocCurveDisplay.from_predictions(y_test, y_prob)

You are likely to see that the model does not tend to predict that individuals survived the voyage. Let's see if we can determine why.

Q7. Look at the average age of survivors vs. those that did not survive the voyage. What do you find?

In [ ]:
# Average age by survival group
titanic.groupby('Survived')['Age'].mean()

Q8. Perhaps the actual age number is as important as whether a passenger was a child or an adult. Create a Boolean column that indicates whether a passenger was a child or an adult. Pick a cutoff age and if they were less than this age, label as a child. We'll see later on how we can make choosing the age part of our model building pipeline, but for now, choose an age or do some calculations to pick an age that you think might do well.

In [ ]:
# Create child indicator (True = child, False = adult)
titanic['is_child'] = titanic['Age'] < 18

titanic[['Age', 'is_child']].head(10)

Q9. When imputing the missing values, you'll need to change the strategy from mean to most_frequent. Impute the missing values, then fit the model using this new variable, generate predictions, and view the confusion matrix. How did your model do?

In [ ]:
titanic['is_child'] = np.where(
    titanic['Age'].isna(),
    np.nan,
    titanic['Age'] < 18
)

# Feature and target
X = titanic[['is_child']].astype(object)
y = titanic['Survived']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=300,
    stratify=y
)

# Imputer
imputer = SimpleImputer(strategy='most_frequent')

# Fit and transform
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Fit model
model = LogisticRegression()
model.fit(X_train_imputed, y_train)

# Predictions
y_pred = model.predict(X_test_imputed)

# Confusion matrix
confuse_m = confusion_matrix(y_test, y_pred)
confuse_m

The confusion matrix shows that the model still predicts all passengers as not surviving (0). While the `is_child` feature was introduced, the model did not improve and still fails to identify any survivors.

This suggests that although being a child may have some influence on survival, it is not a strong enough predictor on its own. The model continues to favor the majority class (non-survivors), resulting in no true positives.

Other common metrics for classification include accuracy, precision, and recall. For more information, see https://developers.google.com/machine-learning/crash-course/classification/accuracy-precision-recall. These metrics can be viewed using the classification_report function.

Q10. Generate a classification report for your model.

In [ ]:
from sklearn.metrics import classification_report

# Generate classification report
print(classification_report(y_test, y_pred, zero_division=0))

The classification report shows that the model performs well for predicting non-survivors (class 0), with a recall of 1.00, meaning it correctly identifies all non-survivors. However, the model completely fails to identify survivors (class 1), with both precision and recall equal to 0. This indicates that the model does not predict any passengers as survivors.Although the overall accuracy is 0.62, this is misleading because the model is biased toward the majority class. Overall, the model performs poorly in identifying the minority class and is not effective for predicting survival.

Another important variable is the Sex variable.

Q11. Create a DataFrame, X, which contains one column, Sex, and a Series, y, which contains the target variable. Perform a 70/30 train/test split, stratifying by the target variable. Make sure that you set a random_state during this split.

In [ ]:
# Feature and target
X = titanic[['Sex']]
y = titanic['Survived']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=300,
    stratify=y
)

X_train.head()

If you inspect the Sex variable, you'll find that it is encoded as a string. We need to convert it to a numeric type so that our model can use it. We can do this using a OneHotEncoder.

Q12. Fit a OneHotEncoder to the training data and then transform both the training and test sets. Fit a model and evaluate the performance of this model.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Create encoder
encoder = OneHotEncoder(drop='first', sparse_output=False)

# Fit on training data and transform
X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

# Fit model
model = LogisticRegression()
model.fit(X_train_encoded, y_train)

# Predictions
y_pred = model.predict(X_test_encoded)

# Evaluation
confuse_m = confusion_matrix(y_test, y_pred)
print(confuse_m)

print(classification_report(y_test, y_pred))

The confusion matrix shows that the model correctly predicts both non-survivors and survivors. It correctly identified 143 non-survivors and 72 survivors. There are 22 false positives and 31 false negatives. Overall, the model performs well and shows a significant improvement compared to earlier models.

Notice how we have used different types of transformations, an imputer for the age-related variables and a OneHotEncoder for the Sex variable. When we need to use multiple types of transformations, a ColumnTransformer is useful.

In the next part, we'll fit a model using both the Sex column and our column indicating whether the passenger was a child.

Create a DataFrame, X, which contains two columns, Sex and the child indicator column, and a Series, y, which contains the target variable. Perform a 70/30 train/test split, stratifying by the target variable. Make sure that you set a random_state during this split. Then create a ColumnTransformer which will apply a SimpleImputer to the child column and a OneHotEncoder to the Sex column. Fit it on the training data and then transform the test data. Then fit a model on this and see how well it does.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Features and target
X = titanic[['Sex', 'is_child']]
y = titanic['Survived']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=300, stratify=y
)

# ColumnTransformer
preprocessor = ColumnTransformer([
    ('child', SimpleImputer(strategy='most_frequent'), ['is_child']),
    ('sex', OneHotEncoder(drop='first'), ['Sex'])
])

# Transform data
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

# Model
model = LogisticRegression()
model.fit(X_train_t, y_train)

# Predictions + evaluation
y_pred = model.predict(X_test_t)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Using both `Sex` and `is_child` with a ColumnTransformer produced results that are identical to the model using `Sex` alone. The confusion matrix and classification metrics remain unchanged, indicating that the `is_child` feature does not provide additional predictive value beyond what is already captured by the `Sex` variable.

This suggests that `Sex` is the dominant predictor of survival, and adding the child indicator does not significantly improve model performance.

Finally, let's see how we can do all of this in a slightly more compact manner. We can use a Pipeline which allows us to specify preprocessing steps (like the ColumnTransformer we used earlier) as well as the model we want to fit.
Q14. Create a Pipeline which contains the ColumnTransformer and the LogisticRegression model.

In [ ]:
# Features and target
X = titanic[['Sex', 'is_child']]
y = titanic['Survived']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=300,
    stratify=y
)

# Preprocessing
preprocessor = ColumnTransformer([
    ('child', SimpleImputer(strategy='most_frequent'), ['is_child']),
    ('sex', OneHotEncoder(drop='first'), ['Sex'])
])

# Pipeline
pipe = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression())
])

# Fit and predict
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

# Evaluation
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

The final model achieved about 80% accuracy. It correctly predicted both survivors and non-survivors, with strong precision and recall for both classes. The biggest improvement came from using the Sex variable, which turned out to be a strong predictor. I also used a pipeline to combine preprocessing and modeling into a clean and reproducible workflow.